In [1]:
import os

# Always run this first — keeps mlruns in the project root, not in notebooks/
os.chdir("..")

print("Working directory:", os.getcwd())


Working directory: c:\ml-ops-tutorial\mlops-tutorial


In [2]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


c:\Users\m6sbhatt\AppData\Local\anaconda3\envs\mlops-env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
url = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases"
    "/heart-disease/processed.cleveland.data"
)
column_names = [
    "age", "sex", "cp", "trestbps", "chol", "fbs",
    "restecg", "thalach", "exang", "oldpeak",
    "slope", "ca", "thal", "target"
]
df = pd.read_csv(url, names=column_names, na_values='?').dropna()
df['target'] = (df['target'] > 0).astype(int)

X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")


Training samples: 237
Test samples:     60


In [4]:
mlflow.set_experiment("heart-disease-classification")

<Experiment: artifact_location='file:///c:/ml-ops-tutorial/mlops-tutorial/mlruns/1', creation_time=1777673158551, experiment_id='1', last_update_time=1777673158551, lifecycle_stage='active', name='heart-disease-classification', tags={}, trace_location=None, workspace='default'>

In [5]:
def run_logistic_regression(C, max_iter, run_name):
    """Train a logistic regression and log everything to MLflow."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_type", "LogisticRegression")
        mlflow.log_param("C", C)
        mlflow.log_param("max_iter", max_iter)
        mlflow.log_param("solver", "lbfgs")

        model = LogisticRegression(
            C=C, max_iter=max_iter, solver='lbfgs', random_state=42
        )
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.sklearn.log_model(model, "model")

        print(f"{run_name}: accuracy={accuracy:.4f}, roc_auc={roc_auc:.4f}")

# Two new logistic regression variants
run_logistic_regression(C=0.01, max_iter=200, run_name='lr-C0.01')
run_logistic_regression(C=10.0, max_iter=200, run_name='lr-C10.0')


2026/05/04 10:35:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:35:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/04 10:35:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:35:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_p

lr-C0.01: accuracy=0.8667, roc_auc=0.9509
lr-C10.0: accuracy=0.8333, roc_auc=0.9487


In [ ]:
def run_gradient_boosting(n_estimators, learning_rate, max_depth, run_name):
    """Train a gradient boosting model and log everything to MLflow."""
    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("model_type", "GradientBoosting")
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_depth", max_depth)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            random_state=42
        )
        model.fit(X_train_scaled, y_train)

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        accuracy = accuracy_score(y_test, y_pred)
        roc_auc = roc_auc_score(y_test, y_proba)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.sklearn.log_model(model, "model")

        print(f"{run_name}: accuracy={accuracy:.4f}, roc_auc={roc_auc:.4f}")

# Two gradient boosting variants
run_gradient_boosting(n_estimators=100, learning_rate=0.1,  max_depth=3, run_name='gb-100-lr0.1')
run_gradient_boosting(n_estimators=200, learning_rate=0.05, max_depth=4, run_name='gb-200-lr0.05')



2026/05/04 10:37:43 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:37:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gb-100-lr0.1: accuracy=0.7667, roc_auc=0.8828


2026/05/04 10:37:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 10:37:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gb-200-lr0.05: accuracy=0.8333, roc_auc=0.9085


In [7]:
client = MlflowClient()

# Get the experiment object so we have its ID
experiment = client.get_experiment_by_name("heart-disease-classification")
experiment_id = experiment.experiment_id

# Search all runs, sorted by roc_auc descending, take the top one
best_run = client.search_runs(
    experiment_ids=[experiment_id],
    order_by=["metrics.roc_auc DESC"],
    max_results=1
)[0]

best_run_id = best_run.info.run_id
best_roc_auc = best_run.data.metrics['roc_auc']
best_run_name = best_run.data.tags.get('mlflow.runName', 'unnamed')

print(f"Best run:     {best_run_name}")
print(f"Run ID:       {best_run_id}")
print(f"ROC AUC:      {best_roc_auc:.4f}")


Best run:     logistic-C0.1
Run ID:       ac6fb4c3eba449d2868885f92f39bf6e
ROC AUC:      0.9520


In [8]:
import time

# Build the artifact URI — points to the model folder inside the run
model_uri = f'runs:/{best_run_id}/model'

# Register it — creates heart-disease-classifier version 1
# (or the next version number if the model name already exists)
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="heart-disease-classifier"
)

# Registration is slightly async — give it a moment to finalize
time.sleep(2)

version = registered_model.version
print(f"Registered as: heart-disease-classifier version {version}")


Successfully registered model 'heart-disease-classifier'.
2026/05/04 10:48:48 WARNING mlflow.tracking._model_registry.fluent: Run with id ac6fb4c3eba449d2868885f92f39bf6e has no artifacts at artifact path 'model', registering model based on models:/m-626a20a50eff4197add58d27be72fd9e instead
Created version '1' of model 'heart-disease-classifier'.


Registered as: heart-disease-classifier version 1


In [9]:
# Assign the champion alias to the version we just registered
# This is the MLflow way of saying: this is the live production model
client.set_registered_model_alias(
    name="heart-disease-classifier",
    alias="champion",
    version=version
)

print(f"heart-disease-classifier v{version} is now @champion")


heart-disease-classifier v1 is now @champion


In [10]:
# Load the model currently aliased as champion
# This exact line is what our FastAPI endpoint will use in Tutorial 7
model = mlflow.sklearn.load_model("models:/heart-disease-classifier@champion")

# Confirm it works by running predictions
y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba)

print("Loaded: models:/heart-disease-classifier@champion")
print(f"Accuracy: {accuracy:.4f}")
print(f"ROC AUC:  {roc_auc:.4f}")


Loaded: models:/heart-disease-classifier@champion
Accuracy: 0.8500
ROC AUC:  0.9520


In [11]:
# Load version 1 specifically — works regardless of its current alias
model_v1 = mlflow.sklearn.load_model("models:/heart-disease-classifier/1")
print("Version 1 loaded successfully")


Version 1 loaded successfully
